# ChatPPT Notebook v0.1

任务目标：
1. 将原始输入中使用的 MasterTemplate 布局名称替换为 `Fair frames presentation.pptx` 中的布局名称。
2. 基于 Fair Frames 模板生成新的 PPT 文件。
3. 自动验证生成结果中的内容与布局是否匹配。

In [6]:
import os
import sys
from pathlib import Path
from pptx import Presentation

ROOT = Path('.').resolve()
SRC_DIR = ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from input_parser import parse_input_text
from ppt_generator import generate_presentation
from template_manager import load_template, get_layout_mapping

MASTER_TEMPLATE = 'templates/MasterTemplate.pptx'
FAIR_TEMPLATE = 'templates/Fair frames presentation.pptx'
print('Environment ready')

Environment ready


- 原始输入：布局名称基于 MasterTemplate

In [ ]:

input_text_master = """
# ChatPPT_Demo_FairFrames

## ChatPPT Demo [Title Only]

## 2024 业绩概述 [Title and Content]
- 总收入增长15%
- 市场份额扩大至30%

## 业绩图表 [Title and Picture 1]
![业绩图表](images/performance_chart.png)

## 新产品发布 [Title and 2 Column]
- 产品A: 特色功能介绍
- 产品B: 市场定位
![未来增长](images/forecast.png)
""".strip()

# 基于占位符结构得到的布局替换映射
layout_name_replacement = {
    'Title Only': 'Title',               # [1] -> [1]
    'Title and Content': 'Thank you',   # [1,7] -> [1,7]
    'Title and Picture 1': 'Section Header 1',  # [1,18] -> [18,1]
    'Title and 2 Column': 'Summary',    # [1,7,18] -> [1,13,7,18]
}

def replace_layout_names(input_text: str, replacement_map: dict[str, str]) -> str:
    updated = input_text
    # 用更长字符串优先替换，避免子串冲突
    for old_name in sorted(replacement_map.keys(), key=len, reverse=True):
        new_name = replacement_map[old_name]
        updated = updated.replace(f'[{old_name}]', f'[{new_name}]')
    return updated

input_text_fair = replace_layout_names(input_text_master, layout_name_replacement)
print(input_text_fair)

# ChatPPT_Demo_FairFrames

## ChatPPT Demo [Title]

## 2024 业绩概述 [Thank you]
- 总收入增长15%
- 市场份额扩大至30%

## 业绩图表 [Section Header 1]
![业绩图表](images/performance_chart.png)

## 新产品发布 [Summary]
- 产品A: 特色功能介绍
- 产品B: 市场定位
![未来增长](images/forecast.png)


- 生成 PPT（使用 Fair Frames 模板）

In [ ]:
fair_prs = load_template(FAIR_TEMPLATE)
fair_layout_mapping = get_layout_mapping(fair_prs)
powerpoint_data, presentation_title = parse_input_text(input_text_fair, fair_layout_mapping)

output_dir = ROOT / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)
output_pptx = output_dir / f'{presentation_title}.pptx'

generate_presentation(powerpoint_data, FAIR_TEMPLATE, str(output_pptx))
print(f'Generated: {output_pptx}')

所有默认幻灯片已被移除。
Presentation saved to '/Users/elon/code/agents-bootcamp/chatppt/outputs/ChatPPT_Demo_FairFrames.pptx'
Generated: /Users/elon/code/agents-bootcamp/chatppt/outputs/ChatPPT_Demo_FairFrames.pptx


- 验证：内容与布局是否匹配

In [ ]:
generated = Presentation(str(output_pptx))

expected_slide_count = len(powerpoint_data.slides)
actual_slide_count = len(generated.slides)
assert actual_slide_count == expected_slide_count, (expected_slide_count, actual_slide_count)

for i, expected in enumerate(powerpoint_data.slides):
    slide = generated.slides[i]
    expected_layout_name = fair_prs.slide_layouts[expected.layout].name
    actual_layout_name = slide.slide_layout.name

    # 1) 校验标题
    actual_title = slide.shapes.title.text if slide.shapes.title else ''
    assert actual_title == expected.content.title, f'Slide {i+1} title mismatch: {actual_title} != {expected.content.title}'

    # 2) 校验布局名
    assert actual_layout_name == expected_layout_name, (
        f'Slide {i+1} layout mismatch: {actual_layout_name} != {expected_layout_name}'
    )

    # 3) 校验要点文本是否出现
    all_text = '\n'.join([shape.text for shape in slide.shapes if getattr(shape, 'has_text_frame', False)])
    for bullet in expected.content.bullet_points:
        assert bullet in all_text, f'Slide {i+1} missing bullet: {bullet}'

    # 4) 如有图片，校验图片占位符中实际存在图片
    if expected.content.image_path:
        has_image = False
        for shape in slide.shapes:
            if not getattr(shape, 'is_placeholder', False):
                continue
            try:
                if int(shape.placeholder_format.type) == 18:
                    _ = shape.image  # 未插图时这里会抛异常
                    has_image = True
                    break
            except Exception:
                pass
        assert has_image, f'Slide {i+1} expected image but none found in picture placeholder'

print('Verification passed: content and layouts match expected mapping.')
print('Output file:', output_pptx)

Verification passed: content and layouts match expected mapping.
Output file: /Users/elon/code/agents-bootcamp/chatppt/outputs/ChatPPT_Demo_FairFrames.pptx


In [10]:
# 可选：打印最终每页布局与标题
for idx, slide in enumerate(generated.slides, start=1):
    title = slide.shapes.title.text if slide.shapes.title else '(no title)'
    print(f'Slide {idx}: layout={slide.slide_layout.name} | title={title}')

Slide 1: layout=Title | title=ChatPPT Demo
Slide 2: layout=Thank you | title=2024 业绩概述
Slide 3: layout=Section Header 1 | title=业绩图表
Slide 4: layout=Summary | title=新产品发布
